In [2]:
import pandas as pd
from ast import literal_eval
import matplotlib.pyplot as plt
import numpy as np
import os
import re
import json
import glob
from ast import literal_eval
from statistics import mean
plt.rcParams["font.sans-serif"]=["SimHei"] #设置字体
plt.rcParams["axes.unicode_minus"]=False #该语句解决图像中的“-”负号的乱码问题
import matplotlib
print(matplotlib.matplotlib_fname())
print(matplotlib.get_cachedir())

/root/anaconda3/envs/llumnix/lib/python3.10/site-packages/matplotlib/mpl-data/matplotlibrc
/root/.cache/matplotlib


将instance.csv中的profiling_data分为开为4列

In [11]:
def get_profiling_data(filename,):
    instance_log = pd.read_csv(filename)
    # 删除dispatch_load_metric为-inf的行
    instance_log = instance_log[instance_log['dispatch_load_metric'] != -np.inf]

    # 将profiling_data列(inference_type,num_seqs,running_seq_lens,last_inference_latency)中的内容转化为4列
    instance_log[['profiling_inference_type', 'profiling_num_seqs', 'running_seq_lens', 'last_inference_latency']] = (
        instance_log['profiling_data']
        .apply(lambda x: literal_eval(x) if pd.notnull(x) else ("", None, None, None))
        .apply(pd.Series)
    )

    instance_log_group = instance_log.groupby("instance_id")
    # 一个group保存为一个sheet
    with pd.ExcelWriter(filename.replace('.csv', '_metrics.xlsx')) as writer:
        for i, (instance_id, group) in enumerate(instance_log_group):
            group.to_excel(writer, sheet_name=f'Instance_{instance_id}', index=False)

# get_profiling_data('/workspace/llm-serve/Llumnix/benchmark_test/logs/A6000-t2-pdd-4/llama-7b/poisson/serve_4_tp1_1000_qps_6_instance.csv')
get_profiling_data('/workspace/llm-serve/Llumnix/benchmark_test/logs/L40-multi-port-zmp-pdd-hetero/llama-7b/poisson/serve_pdd_2000_qps_6_1,1_2_instance.csv')

/root/anaconda3/envs/llumnix/lib/python3.10/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


从per_token_latency_breakdown_list提取中提取每个请求prefill（第一个token）时的各步骤时延

In [2]:

num_req = 2000
qps=12
json_file = f'/workspace/llm-serve/Llumnix/benchmark_test/logs/A6000-t2-multi-port-zmp-pdd-4/llama-7b/poisson/benchmark_4_tp1_{num_req}_qps_{qps}_latency_info.json'
json_file = '/workspace/llm-serve/Llumnix/benchmark_test/logs/l40-pdd-hetero/llama-7b/poisson/benchmark_pdd_2000_qps_6_1,1_2_latency_info.json'
json_file = '/workspace/llm-serve/Llumnix/benchmark_test/logs/l40-pdd-4/llama-7b/poisson/benchmark_pdd_tp1_2000_qps_6_2_2_latency_info.json'
with open(json_file, 'r') as f:
    data = json.load(f)
assert len(data) == 1, "Expected data to contain only one entry"
per_token_latency_breakdown_list = data[0]['per_token_latency_breakdown_list']

# rows保存每个请求的prefill相关的数据
rows = []
decode_no = 10
for req_idx in range(num_req):
    # 将per_token_latency_breakdown_list[req_idx]的第一行(prefill相关)加入
    tmp = min(len(per_token_latency_breakdown_list[req_idx])-1, decode_no)
    rows.append(per_token_latency_breakdown_list[req_idx][tmp])
df = pd.DataFrame(rows)
df.to_csv(json_file.replace('.json', f'_decode_{decode_no}.csv'), index=True)
df.head(n=128)
    # df.to_csv(os.path.join('/workspace/llm-serve/Llumnix/logs/l40-pdd--2/llama-2-7b/uniform/' + migrate_backend, f'request_{req_idx}_timestamps.csv'), index=True)

,api_server_generate_timestamp,manager_generate_timestamp,llumlet_generate_timestamp,engine_add_request_timestamp,engine_process_model_outputs_timestamp_begin,engine_process_model_outputs_timestamp_end,engine_step_timestamp_begin,engine_step_timestamp_end,engine_step_postprocess_timestamp_end,engine_put_queue_timestamp,...,process_model_outputs_latency,engine_step_latency,step_postprocess_latency,across_async_put_queue_thread_latency,across_async_put_queue_actor_latency,across_queue_client_latency,queue_rpc_latency,api_server_get_queue_latency,across_request_streams_latency,migrate_out_one_request_latency
0,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,...,0.707626,27.590275,0.081539,1.849651,3.874302,0.078678,3.512383,0.018835,0.556231,231.339216
1,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,...,0.647068,26.107073,0.108719,2.625465,2.969742,0.028849,3.548622,0.043631,0.829697,396.606207
2,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,...,1.549959,28.502703,0.380278,1.958132,4.316807,0.074148,4.809380,0.032187,0.830412,128.781080
3,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,...,0.589371,25.045872,0.084877,1.746178,4.049301,0.072002,4.313946,0.020266,0.630617,175.881147
4,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,...,0.473976,24.256945,0.043869,0.842333,3.191233,0.085592,5.806446,0.025034,1.154661,304.508924
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,...,0.790596,25.657892,0.093460,1.875877,3.891468,0.071287,4.468679,0.030518,0.883579,130.295992
124,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,...,5.547285,63.233614,0.216484,5.958796,16.303539,0.144482,20.836115,0.123024,7.948399,116.924524
125,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,...,2.316713,34.754038,0.102758,3.113031,4.975557,0.082970,11.132479,0.041962,2.341270,213.441372
126,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,1.750096e+09,...,4.204988,43.180227,0.137806,4.591703,5.545139,0.097990,10.809898,0.076771,5.505800,58.179617


获取请求的各个token时延和迁移时间

In [4]:
import re
def plot_single(ax, latencies):
    hist, bin_edges = np.histogram(latencies, bins=50)
    cumsum = np.cumsum(hist)
    p50 = np.percentile(latencies, 50)
    p80 = np.percentile(latencies, 80)
    p95 = np.percentile(latencies, 95)
    p99 = np.percentile(latencies, 99)
    p999 = np.percentile(latencies, 99.9)
    ax.plot(bin_edges[1:], cumsum/np.sum(hist)*100, color='red')
    ax.axvline(p50, color='blue', linestyle='--', label='P50')
    ax.text(p50, ax.get_ylim()[0] + 0.05 * (ax.get_ylim()[1] - ax.get_ylim()[0]), f"{p50:.2f}", va='bottom', ha='right', color='blue')
    ax.axvline(p80, color='green', linestyle='--', label='P80')
    ax.text(p80, ax.get_ylim()[0] + 0.10 * (ax.get_ylim()[1] - ax.get_ylim()[0]), f"{p80:.2f}", va='bottom', ha='right', color='green')
    ax.axvline(p95, color='orange', linestyle='--', label='P95')
    ax.text(p95, ax.get_ylim()[0] + 0.15 * (ax.get_ylim()[1] - ax.get_ylim()[0]), f"{p95:.2f}", va='bottom', ha='right', color='orange')
    ax.axvline(p99, color='purple', linestyle='--', label='P99')
    ax.text(p99, ax.get_ylim()[0] + 0.20 * (ax.get_ylim()[1] - ax.get_ylim()[0]), f"{p99:.2f}", va='bottom', ha='right', color='purple')
    ax.axvline(p999, color='gray', linestyle='--', label='P99.9')
    ax.text(p999, ax.get_ylim()[0] + 0.25 * (ax.get_ylim()[1] - ax.get_ylim()[0]), f"{p999:.2f}", va='bottom', ha='right', color='gray')
    mean = np.mean(latencies)
    mean_value = bin_edges[:-1][np.where(bin_edges[:-1] <= mean)][-1]
    mean_percentage = cumsum[np.where(bin_edges[:-1] <= mean)][-1] / np.sum(hist) * 100
    ax.axvline(mean_value, color='black', linestyle='-', label='mean={:.2f}'.format(mean))
    ax.text(mean_value, mean_percentage, f"{mean_percentage:.2f}", va='bottom', ha='right', color='black')
    ax.legend(loc='upper right')
    ax.set_ylabel('Cumulative Percentage(%)')
def extract_migration_times(path):
    # 检查文件是否存在
    if not os.path.isfile(path):
        print(f"File {path} does not exist.")
        return {}
    # 初始化结果字典
    migration_times = {}

    # 正则表达式模式（保持不变）
    pattern = r'migrate request \[(.*?)\].*?cost: (\d+\.\d+) ms'

    # 逐行读取文件（自动处理大文件）
    with open(path, 'r', encoding='utf-8') as file:
        for line in file:
            # 直接处理每行（更高效）
            line = line.strip()  # 移除首尾空白字符
            if 'migrate done' in line and 'cost:' in line:
                match = re.search(pattern, line)
                if match:
                    # 提取请求ID和时间（逻辑不变）
                    ids_str = match.group(1)
                    time = float(match.group(2))
                    request_ids = [req_id.strip("'") for req_id in ids_str.split(', ')]
                    # 更新字典
                    for req_id in request_ids:
                        migration_times[req_id] = time
    return migration_times

def extract_migration_waiting_times(log_file_path):
    # 定义一个字典，用于存储请求 ID 对应的时间戳
    request_timestamps = {}
    migrate_waiting_times = []
    # 打开日志文件并逐行读取
    with open(log_file_path, 'r') as file:
        for line in file:
            # 检查行是否包含 engine_step_timestamp_end 或 _migrate_out_one_request start
            if "engine_step_timestamp_end" in line or "_migrate_out_one_request start" in line:
                # 使用正则表达式提取请求 ID
                request_id_match = re.search(r'[0-9a-f]{32}', line)
                # 使用正则表达式提取时间戳
                timestamp_match = re.search(r'timestamps: \d+\.\d+', line)

                if request_id_match and timestamp_match:
                    request_id = request_id_match.group()
                    timestamp = float(timestamp_match.group().split(":")[1])

                    # 如果请求 ID 不在字典中，则初始化一个条目
                    if request_id not in request_timestamps:
                        request_timestamps[request_id] = {
                            "engine_step_timestamp_end": None,
                            "_migrate_out_one_request start": None
                        }

                    # 根据日志行内容更新对应的时间戳
                    if "engine_step_timestamp_end" in line:
                        request_timestamps[request_id]["engine_step_timestamp_end"] = timestamp
                    elif "_migrate_out_one_request start" in line:
                        request_timestamps[request_id]["_migrate_out_one_request start"] = timestamp
                    if request_timestamps[request_id]["_migrate_out_one_request start"] is not None and request_timestamps[request_id]["engine_step_timestamp_end"] is not None:
                        request_timestamps[request_id]["migrate_waiting_time"] = (request_timestamps[request_id]["_migrate_out_one_request start"] - request_timestamps[request_id]["engine_step_timestamp_end"]) *1000
                        request_timestamps[request_id]["migrate_waiting_time"] = max(0, request_timestamps[request_id]["migrate_waiting_time"])
                        migrate_waiting_times.append(request_timestamps[request_id]["migrate_waiting_time"])
    
    # if len(migrate_waiting_times) > 0:
    #     fig, (ax) = plt.subplots(1, 1, figsize=(7, 4.8))
    #     fig.suptitle(log_file_path, fontsize=14)
    #     plot_single(ax, migrate_waiting_times)
        
    return request_timestamps


In [ ]:

# 读取一个json文件的指定字段
def json_to_decode_latencies(json_file, csv_file, log_file=None):
    with open(json_file, 'r') as f:
        data = json.load(f)
    # 提取指定字段
    token_latencies_list = data[0]['all_decode_token_latencies']
    response_lens = data[0]['request_lens']
    # token_latencies_list是一个list，response_lens保存了每个请求在token_latencies_list中对应的长度，基于response_lens将其转换为二维数组
    output_len_per_seq = max(response_lens)
    token_latencies = np.zeros((len(response_lens), output_len_per_seq))

    t=0
    for i in range(len(response_lens)):
        token_latencies[i][:response_lens[i]] = token_latencies_list[t:t+response_lens[i]]
        # assert token_latencies[i][0] != token_latencies[i][1], f"token_latencies[{i}][0]== token_latencies[{i}][1] = {token_latencies[i][0]} == {token_latencies[i][1]}"
        if token_latencies[i][0] == token_latencies[i][1]:
            print(f"token_latencies[{i}][0] == token_latencies[{i}][1] = {token_latencies[i][0]} == {token_latencies[i][1]}")
        t += response_lens[i]
    
    # 将数据转换为DataFrame，将request_ids作为index
    df = pd.DataFrame(token_latencies, index=data[0]['request_ids'])

    # 设置列名为token_id
    df.columns = [f'decode_token_{i+1}' for i in range(output_len_per_seq)]

    # 获取迁移时间
    if 'pdd' in json_file.split('/')[-1]:
        if log_file is None:
            # 将'/workspace/llm-serve/Llumnix/logs/l40-pdd--2/llama-2-7b/poisson/benchmark_pdd_tp1_1000_qps_4_prompt_len_1024_response_len_64_1_1_latency_info.json'转化为'/workspace/llm-serve/Llumnix/logs/l40-pdd--2/llama-2-7b/poisson/benchmark_pdd_tp1_1000_qps_4_1_1_prompt_len_1024_response_len_64_latency_info.json'
            p_d = json_file[-22:-17]    # _1_1_
            tmp = json_file[:-22].split('qps_')
            tmp = tmp[0] + 'qps_' + tmp[1][0] + p_d + tmp[1][2:] + json_file[-17:]
            log_path = tmp.replace('_latency_info.json', '.log')
            # 将最后一个'benchmark'替换为'serve'
            if 'benchmark' in log_path:
                log_path = re.sub(r'benchmark(?!.*benchmark)', 'serve', log_path)
            print(log_path)
            # log_path = log_path.replace('benchmark', 'serve')
        else:
            log_path = log_file
        
        migration_times = extract_migration_times(log_path)
        migration_waiting_times = extract_migration_waiting_times(log_path)
        # print(log_path,len(migration_times))

        # 在df中添加一列，列名为migration_time，放在第一列
        df.insert(0, 'migration_time', 0.0)
        df.insert(0, 'migration_waiting_time', 0.0)
        # 遍历df的index，获取对应的migration_time
        for index in df.index:
            # 如果index在migration_times中，则将对应的值赋值给df['migration_time']
            # print(index, index in migration_times)
            if index in migration_times:
                df.at[index, 'migration_time'] = migration_times[index]

            if index in migration_waiting_times and "migrate_waiting_time" in migration_waiting_times[index]:
                # print(migration_waiting_times[index])
                df.at[index, 'migration_waiting_time'] = migration_waiting_times[index]['migrate_waiting_time']
        # 绘制迁移时间和迁移等待时间的直方图
        if len(df['migration_time']) > 0:
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 4.8))
            fig.suptitle(log_file, fontsize=14)
            plot_single(ax1, df['migration_time'])
            ax1.set_title('Migration Time Distribution')
            plot_single(ax2, df['migration_waiting_time'])
            ax2.set_title('Migration Waiting Time Distribution')
            plt.tight_layout()
            plt.savefig(json_file.replace('.json', '_migration_times.png'))
            plt.close(fig)

    # print(csv_file)
    df.insert(0, 'prefill_time', data[0]['prefill_token_latencies'])
    # 保存为csv文件
    df.to_csv(csv_file, index=True)

# glob.glob(os.path.join(dir, '*.json'))
filenames = [
    # '/workspace/llm-serve/Llumnix/benchmark_test/logs/l40-pdd-hetero/llama-7b/poisson/benchmark_pdd_2000_qps_6_1,1_2_latency_info.json',
    '/workspace/llm-serve/Llumnix/benchmark_test/logs/L40-multi-port-zmp-pdd-4/llama-7b/poisson/benchmark_4_tp1_2000_qps_6_latency_info.json',
    '/workspace/llm-serve/Llumnix/benchmark_test/logs/L40-multi-port-zmp-pdd-4/llama-7b/poisson/benchmark_4_tp1_2000_qps_10_latency_info.json',
    '/workspace/llm-serve/Llumnix/benchmark_test/logs/L40-multi-port-zmp-pdd-4/llama-7b/poisson/benchmark_4_tp1_2000_qps_12_latency_info.json',
    '/workspace/llm-serve/Llumnix/benchmark_test/logs/L40-multi-port-zmp-pdd-4/llama-7b/poisson/benchmark_pdd_tp1_2000_qps_6_2_2_latency_info.json',
    '/workspace/llm-serve/Llumnix/benchmark_test/logs/L40-multi-port-zmp-pdd-hetero/llama-7b/poisson/benchmark_pdd_2000_qps_6_1,1_2_latency_info.json',
    '/workspace/llm-serve/Llumnix/benchmark_test/logs/L40-multi-port-zmp-pdd-4/llama-7b/poisson/benchmark_pdd_tp1_2000_qps_2_2_2_latency_info.json'
]
log_filenames = [
    None,None,None,None,
    '/workspace/llm-serve/Llumnix/benchmark_test/logs/L40-multi-port-zmp-pdd-hetero/llama-7b/poisson/serve_pdd_2000_qps_6_1,1_2.log',
    None
]
for filename, log_file in zip(filenames, log_filenames):
    
    output_path = filename.replace('latency_info.json', 'all_decode_token_latencies.csv')
    if os.path.exists(output_path):
        continue
    print(filename)
    json_to_decode_latencies(filename, output_path, log_file=log_file)

In [17]:
def extract_migration_info(path):
    '''
    migration_info[req_id] = {
                            "blocks": blocks,
                            "time_ms": migrate time,
                            "speed_blocks_per_s": speed
                            "migrate_waiting_time": migrate_waiting_time(ms)
                        }
    '''
    # 检查文件是否存在
    if not os.path.isfile(path):
        print(f"File {path} does not exist.")
        return {}

    migration_info = {}

    # 示例： Instance ... migrate done, migrate request ['494c45676def4572986621d1afbc337f'], migration status: MigrationStatus.FINISHED, len: 7 blocks, cost: 240.65113067626953 ms
    # 正确的正则表达式应为：
    pattern = r"migrate request \[(.*?)\].*?len: (\d+) blocks,.*?cost: ([\d\.]+) ms"
    count = 0
    # 逐行读取文件（自动处理大文件）
    with open(path, 'r', encoding='utf-8') as file:
        for line in file:
            line = line.strip()
            if 'migrate done' in line and 'cost:' in line:
                if count < 10:
                    count += 1
                    continue
                match = re.search(pattern, line)
                if match:
                    ids_str = match.group(1)
                    blocks = int(match.group(2))
                    time = float(match.group(3))
                    speed = blocks / time * 1000 if time > 0 else 0  # blocks/ms -> blocks/s
                    request_ids = [req_id.strip("'") for req_id in ids_str.split(', ')]
                    for req_id in request_ids:
                        migration_info[req_id] = {
                            "blocks": blocks,
                            "time_ms": time,
                            "speed_blocks_per_s": speed
                        }

    # 求平均speed
    if migration_info:
        avg_speed = mean(info['speed_blocks_per_s'] for info in migration_info.values())
        print(f"Average migration speed: {avg_speed:.2f} blocks/s")
        for req_id, info in migration_info.items():
            info['avg_speed_blocks_per_s'] = avg_speed
    else:
        print("No migration information found.")
    # return migration_info
    migration_waiting_times = extract_migration_waiting_times(path)
    # print(migration_waiting_times)
    for req_id, info in migration_info.items():
        if req_id in migration_waiting_times:
            migration_info[req_id]['migrate_waiting_time'] = migration_waiting_times[req_id]['migrate_waiting_time']
        else:
            print(req_id, "not in migration_waiting_times")
            migration_info[req_id]['migrate_waiting_time'] = 0.0

    print(f'Average migration waiting time: {mean(info["migrate_waiting_time"] for info in migration_info.values()):.2f} ms')
    # print(migration_info)
    return

log_paths = [
    # '/workspace/llm-serve/Llumnix/benchmark_test/logs/L40-multi-port-zmp-pdd-hetero/llama-7b/poisson/serve_pdd_2000_qps_6_1,1_2.log',
    # '/workspace/llm-serve/Llumnix/benchmark_test/logs/A6000-t2-multi-port-zmp-pdd-hetero/llama-7b/poisson/serve_pdd_2000_qps_6_1,1_2.log',
    '/workspace/llm-serve/Llumnix/benchmark_test/logs/L40-multi-port-zmp-pdd-4/llama-7b/poisson/serve_pdd_tp1_2000_qps_6_2_2.log',
    # '/workspace/llm-serve/Llumnix/benchmark_test/logs/L40-multi-port-zmp-parallel-pdd-4/llama-7b/poisson/serve_pdd_tp1_2000_qps_6_1_3.log'
    '/workspace/llm-serve/Llumnix/benchmark_test/logs/L40-multi-port-zmp-parallel-pdd-4/llama-7b/poisson/serve_pdd_tp1_2000_qps_6_2_2.log'
]
for log_path in log_paths:
    print(f"Processing log file: {log_path}")
    if not os.path.exists(log_path):
        print(f"Log file {log_path} does not exist.")
        continue
    # 调用函数提取迁移速度
    extract_migration_info(log_path)

Processing log file: /workspace/llm-serve/Llumnix/benchmark_test/logs/L40-multi-port-zmp-pdd-4/llama-7b/poisson/serve_pdd_tp1_2000_qps_6_2_2.log
Average migration speed: 54.37 blocks/s
 not in migration_waiting_times
Average migration waiting time: 35827.87 ms
Processing log file: /workspace/llm-serve/Llumnix/benchmark_test/logs/L40-multi-port-zmp-parallel-pdd-4/llama-7b/poisson/serve_pdd_tp1_2000_qps_6_2_2.log
Average migration speed: 29.67 blocks/s
 not in migration_waiting_times
Average migration waiting time: 30409.15 ms
